# CNN Car Image Classifier — 7 Classes

**PPR501 — Machine Learning for Engineers (Applied Machine Learning)**

Bài toán được giải quyết trong phần này là **nhận diện ảnh xe ô tô** bằng mạng nơ-ron
tích chập (CNN), với đầu ra là một trong bảy nhóm thương hiệu được liệt kê bên dưới.

Kiến trúc được sử dụng là **MobileNetV2** — một mạng CNN tương đối gọn được huấn
luyện trước trên một tập ảnh rất lớn. Nhờ vậy, chỉ cần một giai đoạn fine-tuning ngắn
là đạt được kết quả tốt, dù tập dữ liệu sử dụng trong đồ án chỉ chứa vài nghìn ảnh.

| # | Class |
|---|-------|
| 0 | Audi |
| 1 | Hyundai Creta |
| 2 | Mahindra Scorpio |
| 3 | Rolls Royce |
| 4 | Maruti Suzuki Swift |
| 5 | Tata Safari |
| 6 | Toyota Innova |

## Hướng dẫn chạy

Notebook **chạy độc lập hoàn toàn** — không phụ thuộc vào bất kỳ file nào khác trong
project; toàn bộ mã nguồn nằm trong chính notebook này và có thể chạy tuần tự từng cell.

**Các thư viện cần cài đặt:** TensorFlow, Keras, NumPy, Matplotlib, Scikit-learn,
Gradio, Pillow.

```bash
pip install tensorflow numpy matplotlib scikit-learn gradio pillow
```

**Tập dữ liệu** phải được đặt tại `sources/Dataset/Train/<Class>/*.jpg` và
`sources/Dataset/Test/<Class>/*.jpg`. Đường dẫn được tự động xác định trong cell bên
dưới, do đó không cần chỉnh sửa mã nguồn.

> ⏱️ Thời gian thực thi toàn bộ quy trình (20 epoch) khoảng **10–15 phút** trên CPU
> thông thường.

In [ ]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from PIL import Image

print(f"TensorFlow {tf.__version__}")

## 1. Cấu hình (Configuration)

Toàn bộ siêu tham số được tập trung tại một vị trí duy nhất, thuận tiện cho việc
tinh chỉnh về sau.

In [ ]:
# ============================================================
# CELL 2 — Configuration
# ============================================================
SEED = 42
IMAGE_SIZE = 224          # MobileNetV2 mặc định 224x224
BATCH_SIZE = 32
NUM_CLASSES = 7
VALIDATION_SPLIT = 0.2   # 20% của Train để làm validation

EPOCHS_FROZEN = 5        # Stage 1: đóng băng backbone
EPOCHS_FINETUNE = 15     # Stage 2: mở băng backbone
LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-4
PATIENCE = 6

CLASS_NAMES = [
    "Audi",
    "Hyundai Creta",
    "Mahindra Scorpio",
    "Rolls Royce",
    "Maruti Suzuki Swift",
    "Tata Safari",
    "Toyota Innova",
]

tf.keras.utils.set_random_seed(SEED)

## 2. Xác định vị trí tập dữ liệu

Đường dẫn tập dữ liệu được **tự động dò theo thứ tự ưu tiên**, nhờ đó notebook vẫn
chạy được khi được di chuyển sang vị trí khác trong cùng project.

In [ ]:
# ============================================================
# CELL 3 — Locate dataset
# ============================================================
def locate_dataset():
    """Tìm thư mục Dataset/Train chứa ảnh xe."""
    here = Path.cwd()
    candidates = [
        here / "Dataset",
        here / "sources" / "Dataset",
        here.parent / "Dataset",
        here.parent / "sources" / "Dataset",
        here.parent.parent / "Dataset",
        here.parent.parent / "sources" / "Dataset",
    ]
    for c in candidates:
        if (c / "Train").is_dir() and (c / "Test").is_dir():
            return c.resolve()
    raise FileNotFoundError(
        "Không tìm thấy dataset.\n"
        "Cần cấu trúc:  <...>/Dataset/Train/<Class>/*.jpg  và  <...>/Dataset/Test/<Class>/*.jpg\n"
        "Đã thử từ: " + ", ".join(str(c) for c in candidates)
    )


DATASET_DIR = locate_dataset()
TRAIN_DIR = DATASET_DIR / "Train"
TEST_DIR = DATASET_DIR / "Test"

print(f"Dataset : {DATASET_DIR}")
print(f"Train   : {TRAIN_DIR}")
print(f"Test    : {TEST_DIR}")

### Kiểm tra tập dữ liệu

Xác nhận sự hiện diện của đủ bảy lớp và khảo sát số lượng ảnh của từng lớp. Các con số
này quyết định cách thức xử lý hiện tượng mất cân bằng ở những phần tiếp theo.

In [ ]:
# ============================================================
# CELL 4 — Dataset overview
# ============================================================
def class_counts(dataset_dir: Path) -> Counter:
    """Đếm số ảnh trong mỗi thư mục lớp."""
    counter = Counter()
    for folder in sorted(p for p in dataset_dir.iterdir() if p.is_dir()):
        counter[folder.name] = sum(1 for f in folder.iterdir() if f.is_file())
    return counter


train_counts = class_counts(TRAIN_DIR)
test_counts = class_counts(TEST_DIR)

print(f"{'Class folder':<22}{'Train':>8}{'Test':>8}")
print("-" * 38)
for name in sorted(train_counts):
    print(f"{name:<22}{train_counts[name]:>8}{test_counts.get(name, 0):>8}")
print("-" * 38)
print(f"{'TOTAL':<22}{sum(train_counts.values()):>8}{sum(test_counts.values()):>8}")

print("\nTrain classes:", sorted(train_counts))
print("Test  classes:", sorted(test_counts))

if sorted(train_counts) != sorted(test_counts):
    print("\n⚠️  Tên thư mục Train và Test KHÔNG khớp nhau — cần kiểm tra lại dataset.")

### Về hiện tượng mất cân bằng dữ liệu

Trong tập huấn luyện, lớp Audi chứa 814 ảnh trong khi lớp Hyundai Creta chỉ chứa 271
ảnh — chênh lệch khoảng 3 lần. Nếu không xử lý, mô hình sẽ thiên lệch rõ rệt về lớp
đông nhất và dự đoán "Audi" cho gần như mọi ảnh đầu vào.

Hai công cụ được áp dụng đồng thời (chi tiết ở nội dung ngay bên dưới và tại phần 7):

1. **Class weight** trong quá trình huấn luyện — các lớp hiếm được gán trọng số cao
   hơn trong hàm loss.
2. **Prior correction** ở bước dự đoán — xác suất đầu ra được hiệu chỉnh lại theo tần
   suất thực tế của từng lớp.

In [ ]:
# ============================================================
# CELL 5 — Class weights (effective number)
# ============================================================
def compute_class_weight(counts: Counter, mode: str = "effective") -> dict:
    """
    Tính trọng số cho từng lớp.

    'effective' dùng công thức effective-number của Cui et al. (2019):
        w_c = (1 - beta) / (1 - beta ** n_c)
    Nhẹ nhàng hơn nhiều so với inverse frequency thuần, phù hợp transfer learning.

    Mọi trọng số được chuẩn hoá để trung bình = 1.0, nếu không gradient sẽ bị co lại
    khoảng 500 lần và model gần như không học được gì trong số epoch cho phép.
    """
    folder_order = sorted(counts)
    total = sum(counts.values())
    n_classes = len(counts)

    if mode == "none":
        return {i: 1.0 for i in range(n_classes)}

    if mode == "balanced":
        raw = {i: total / (n_classes * counts[name]) for i, name in enumerate(folder_order)}
    else:  # effective
        beta = 0.999
        raw = {}
        for i, name in enumerate(folder_order):
            n = counts[name]
            raw[i] = (1.0 - beta) / (1.0 - beta**n) if n > 0 else 1.0

    mean = sum(raw.values()) / len(raw)
    return {i: w / mean for i, w in raw.items()}


class_weight = compute_class_weight(train_counts, mode="effective")

print("Effective-number class weights (mean = 1.0):")
for i, name in enumerate(sorted(train_counts)):
    print(f"  {name:<22} n={train_counts[name]:<5} w={class_weight[i]:.2f}")

print("Priors (tần suất tương đối của mỗi lớp):")
total_train = sum(train_counts.values())
PRIORS = np.array([train_counts[n] / total_train for n in sorted(train_counts)])
for name, p in zip(sorted(train_counts), PRIORS):
    print(f"  {name:<22} p={p:.3f}")

## 3. Chuẩn bị dữ liệu (Data Pipeline)

Yếu tố quan trọng nhất của phần này là **duy trì sự tách biệt giữa tập validation và
tập test**:

- `Dataset/Train` → 80% train + 20% validation
- `Dataset/Test` → chỉ dùng cho đánh giá cuối cùng, **không** dùng để chọn model

Việc sử dụng tập test làm tập validation (lỗi thường gặp) làm cho độ chính xác được
báo cáo trở nên không đáng tin: mô hình đã "từng thấy" chính những ảnh đó trong quá
trình huấn luyện.

In [ ]:
# ============================================================
# CELL 6 — Build tf.data datasets
# ============================================================
def build_datasets(image_size, batch_size, validation_split):
    """
    Tạo train / validation / test dataset.

    Dataset/Train được chia thành train (80%) và validation (20%).
    Dataset/Test chỉ dùng cho đánh giá cuối.
    Augmentation chỉ áp dụng cho tập train.
    """
    common = dict(
        image_size=(image_size, image_size),
        batch_size=batch_size,
        labels="inferred",
    )

    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, subset="training", seed=SEED, shuffle=True,
        validation_split=validation_split, **common
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, subset="validation", seed=SEED, shuffle=True,
        validation_split=validation_split, **common
    )
    test_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, shuffle=False, **common
    )

    # Chuẩn hoá về [0, 1] cho MỌI tập.
    normalize = tf.keras.Sequential(
        [tf.keras.layers.Rescaling(1.0 / 255.0)], name="normalize"
    )

    # Augmentation nhẹ, chỉ cho train. Xe đối xứng nên flip ngang hợp lý,
    # nhưng xoay/zoom mạnh sẽ cắt mất chi tiết phân biệt (lưới tản nhiệt, huy hiệu).
    augment = tf.keras.Sequential(
        [
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.10),
            tf.keras.layers.RandomZoom(height_factor=0.10, width_factor=0.10),
            tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
            tf.keras.layers.RandomBrightness(factor=0.10),
        ],
        name="augmentation",
    )

    train_ds = train_ds.map(
        lambda x, y: (normalize(augment(x, training=True)), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    val_ds = val_ds.map(
        lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE
    )
    test_ds = test_ds.map(
        lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE
    )

    return tuple(
        d.prefetch(tf.data.AUTOTUNE) for d in (train_ds, val_ds, test_ds)
    )


train_ds, val_ds, test_ds = build_datasets(IMAGE_SIZE, BATCH_SIZE, VALIDATION_SPLIT)

print(f"Train batches: {train_ds.cardinality().numpy()}")
print(f"Val   batches: {val_ds.cardinality().numpy()}")
print(f"Test  batches: {test_ds.cardinality().numpy()}")

sample_x, _ = next(iter(train_ds))
print(f"\nPixel range: [{sample_x.numpy().min():.4f}, {sample_x.numpy().max():.4f}]  (phải là [0,1])")

## 4. Kiến trúc mô hình (Model Architecture)

### Đối chiếu hai kiến trúc CNN

| | `scratch` (CNN tự xây dựng) | `mobilenetv2` (dùng cho mô hình chính) |
|---|---|---|
| Bản chất | CNN 3 tầng tích chập tự viết | CNN 35 tầng tích chập |
| Số tham số | ~103 K | ~2.4 M |
| Nguồn trọng số | Khởi tạo ngẫu nhiên | Học trước trên 1.2 triệu ảnh ImageNet |
| Kết quả | Sụp về lớp đông nhất (Audi) | Cân bằng, accuracy ~0.97 |

Phiên bản `scratch` được giữ lại nhằm **minh hoạ** rằng 3 352 ảnh là không đủ để huấn
luyện CNN từ đầu — đây chính là lý do lựa chọn MobileNetV2.

### Cơ sở lựa chọn MobileNetV2

Tập dữ liệu chỉ chứa **3 352 ảnh**, con số quá nhỏ để huấn luyện một mạng CNN từ đầu.

Trong thử nghiệm với một CNN cỡ nhỏ được khởi tạo ngẫu nhiên, mô hình học được một vài
epoch rồi *sụp* về lớp đông nhất — dự đoán "Audi" cho mọi ảnh. Đây là hậu quả của
overfitting kết hợp với mất cân bằng dữ liệu.

**Transfer learning** giải quyết đúng nguyên nhân đó: MobileNetV2 đã được huấn luyện
trên **1.2 triệu ảnh ImageNet**, nên đã nắm được các đặc trưng cơ bản về hình khối, kết
cấu bề mặt và vật liệu. Chỉ với vài epoch fine-tuning, mô hình đã đạt được độ cân
bằng giữa các lớp.

### So sánh số tham số

| Kiến trúc | Số tham số | Nhận xét |
|-----------|-----------|---------|
| CNN từ đầu (Flatten + Dense) | ~11.8 M | 99.9% tập trung trong một lớp Dense, overfitting nghiêm trọng |
| MobileNetV2 + head nhỏ | ~2.4 M | Phần lớn là backbone đã huấn luyện trước |

In [ ]:
# ============================================================
# CELL 7 — Build MobileNetV2 model
# ============================================================
def build_model(num_classes=NUM_CLASSES, image_size=IMAGE_SIZE):
    """
    MobileNetV2 pre-trained trên ImageNet + classification head mới.

    include_top=False bỏ classifier gốc, pooling='avg' gom feature map thành
    vector 1280 chiều thay vì Flatten thành hàng trăm nghìn — đó là lý do
    tham số giảm mạnh và model tổng quát hoá tốt hơn.
    """
    base = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(image_size, image_size, 3),
        pooling="avg",
    )
    base.trainable = False  # Stage 1: đóng băng backbone

    return tf.keras.Sequential(
        [
            base,
            layers.Dropout(0.30),
            layers.Dense(128, activation="relu"),
            layers.Dropout(0.20),
            layers.Dense(num_classes, activation="softmax", name="predictions"),
        ],
        name="car_mobilenetv2",
    )


model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_STAGE1),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Kết quả in ra từ `model.summary()`:

- **Total params ~2.42 M** — nhỏ hơn đáng kể so với CNN huấn luyện từ đầu.
- **Trainable params chỉ 164 871** — do backbone đang ở trạng thái đóng băng.
- Lớp `predictions (Dense)` có **7** đơn vị đầu ra, tương ứng với 7 lớp.

In [ ]:
# ============================================================
# CELL 8 — Stage 1: train với backbone đóng băng
# ============================================================
checkpoint_path = "best_model.weights.h5"

callbacks_stage1 = [
    ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
    ),
    EarlyStopping(monitor="val_accuracy", patience=PATIENCE, restore_best_weights=True),
]

print(f"\n{'='*60}")
print("STAGE 1/2: Frozen backbone")
print(f"{'='*60}")

history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    class_weight=class_weight,
    callbacks=callbacks_stage1,
)

print(f"\nStage 1 done. Best val_accuracy = {max(history_1.history['val_accuracy']):.4f}")

## 5. Stage 2 — Fine-tune toàn bộ mô hình

Sau khi đầu classifier đã học được các đặc trưng cơ bản, backbone được mở băng để tinh
chỉnh tiếp các đặc trưng ở tầng thấp (cạnh, góc, kết cấu bề mặt).

Tốc độ học được hạ từ `1e-3` xuống `1e-4` nhằm hạn chế sự suy giảm của những kiến thức
đã đạt được ở giai đoạn trước.

> ⚠️ **Lưu ý:** các lớp `BatchNormalization` **không** được mở băng. Thống kê chạy
> (running mean/var) của chúng được ước lượng trên ImageNet; nếu cho phép chúng biến
> động trên chỉ 2 682 ảnh thì các thống kê này sẽ bị phá hủy hoàn toàn.

In [ ]:
# ============================================================
# CELL 9 — Stage 2: unfreeze backbone
# ============================================================
def unfreeze_backbone(model):
    """
    Mở băng backbone nhưng GIỮ NGUYÊN BatchNorm.
    """
    for layer in model.layers:
        layer.trainable = True
    for layer in model.layers:
        sublayers = getattr(layer, "layers", []) or []
        for sub in sublayers:
            sub.trainable = True
            if isinstance(sub, layers.BatchNormalization):
                sub.trainable = False

    bn_frozen = sum(
        1
        for layer in model.layers
        for sub in (getattr(layer, "layers", []) or [])
        if isinstance(sub, layers.BatchNormalization) and not sub.trainable
    )
    return bn_frozen


n_bn = unfreeze_backbone(model)
print(f"Backbone unfrozen ({n_bn} BatchNorm layers kept frozen)")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_STAGE2),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    class_weight=class_weight,
    callbacks=[
        ModelCheckpoint(
            checkpoint_path,
            monitor="val_accuracy",
            save_best_only=True,
            save_weights_only=True,
        ),
        ReduceLROnPlateau(
            monitor="val_accuracy",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
        ),
        EarlyStopping(
            monitor="val_accuracy",
            patience=PATIENCE,
            restore_best_weights=True,
        ),
    ],
)

print(f"\nStage 2 done. Best val_accuracy = {max(history_2.history['val_accuracy']):.4f}")

## 6. Đánh giá trên tập test

Đây là thời điểm mô hình lần đầu tiên được áp dụng lên **tập test**. Tập này chưa từng
tham gia vào bất kỳ quá trình nào sau đây: huấn luyện, validation, dừng sớm (early
stopping) hay lựa chọn checkpoint.

In [ ]:
# ============================================================
# CELL 10 — Evaluate on test set
# ============================================================
def predict_probas(model, dataset):
    """Trả về (y_true, probabilities) cho toàn bộ dataset."""
    y_true, probs = [], []
    for images, labels in dataset:
        probs.append(model.predict(images, verbose=0))
        y_true.extend(labels.numpy().tolist())
    return np.array(y_true), np.concatenate(probs, axis=0)


y_true, probs_raw = predict_probas(model, test_ds)
y_pred = probs_raw.argmax(axis=1)

print("Classification report (WITHOUT prior correction)")
print(classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    zero_division=0,
))

acc_raw = float((y_pred == y_true).mean())
balanced_raw = recall_score(y_true, y_pred, average="macro")
print(f"Test accuracy          : {acc_raw:.4f}")
print(f"Balanced accuracy      : {balanced_raw:.4f}")

## 7. Prior Correction — hiệu chỉnh mất cân bằng

### Đặt vấn đề

Ngay cả khi đã áp dụng class weight, xác suất đầu ra vẫn còn thiên lệch nhẹ về các
lớp phổ biến. Một bước hiệu chỉnh bổ sung được thực hiện theo kỹ thuật **logit
adjustment**:

$$
q_c \propto \frac{p_c}{\text{prior}_c^{\alpha}}
$$

- $\alpha = 0$ → giữ nguyên xác suất gốc
- $\alpha = 1$ → loại bỏ hoàn toàn ảnh hưởng của prior

Giá trị $\alpha$ được xác định trên **tập validation** (hoàn toàn không dùng tới tập
test) bằng cách thử lần lượt các giá trị ứng viên và giữ lại giá trị cho **balanced
accuracy** cao nhất.

In [ ]:
# ============================================================
# CELL 11 — Select alpha on VALIDATION split
# ============================================================
def apply_prior_correction(probs, priors, alpha=1.0):
    """
    Hiệu chỉnh xác suất theo prior của lớp.
    alpha=0 giữ nguyên; alpha=1 loại bỏ hoàn toàn prior.
    """
    if alpha == 0:
        return probs
    p = np.clip(np.asarray(probs, dtype="float64"), 1e-9, 1.0)
    logits = np.log(p) - alpha * np.log(np.clip(priors, 1e-9, 1.0))
    logits -= logits.max(axis=-1, keepdims=True)
    exp = np.exp(logits)
    return (exp / exp.sum(axis=-1, keepdims=True)).astype("float32")


y_val, probs_val = predict_probas(model, val_ds)

print("Tuning alpha on the VALIDATION split:\n")
best_alpha, best_bal = 0.0, recall_score(y_val, probs_val.argmax(axis=1), average="macro")
for alpha in [0.0, 0.25, 0.50, 0.75, 1.00]:
    adj = apply_prior_correction(probs_val, PRIORS, alpha)
    bal = recall_score(y_val, adj.argmax(axis=1), average="macro")
    print(f"  alpha={alpha:.2f} -> val balanced accuracy {bal:.4f}")
    if bal > best_bal:
        best_alpha, best_bal = alpha, bal

print(f"\nSelected prior correction alpha = {best_alpha}")

### Áp dụng giá trị alpha đã chọn lên tập test

Chỉ đến bước này, các xác suất đã hiệu chỉnh mới được sử dụng cho đánh giá cuối cùng.

In [ ]:
# ============================================================
# CELL 12 — Final test evaluation with prior correction
# ============================================================
probs_adj = apply_prior_correction(probs_raw, PRIORS, best_alpha)
y_pred_adj = probs_adj.argmax(axis=1)

print("Classification report (WITH prior correction, alpha={:.2f})".format(best_alpha))
print(classification_report(
    y_true, y_pred_adj,
    target_names=CLASS_NAMES,
    zero_division=0,
))

acc_adj = float((y_pred_adj == y_true).mean())
balanced_adj = recall_score(y_true, y_pred_adj, average="macro")

print(f"Test accuracy          : {acc_adj:.4f}")
print(f"Balanced accuracy      : {balanced_adj:.4f}")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"  Val accuracy         : {max(history_2.history['val_accuracy']):.4f}")
print(f"  Test accuracy        : {acc_adj:.4f}")
print(f"  Balanced accuracy    : {balanced_adj:.4f}")
print(f"  Prior alpha          : {best_alpha}")
print("\nBefore vs after prior correction:")
print(f"  accuracy      {acc_raw:.4f} -> {acc_adj:.4f}")
print(f"  balanced acc  {balanced_raw:.4f} -> {balanced_adj:.4f}")

## 8. Trực quan hoá kết quả

### Đường cong huấn luyện

Đường cong train và validation cần bám sát nhau. Nếu khoảng cách giữa chúng tăng dần
thì mô hình đang overfitting.

In [ ]:
# ============================================================
# CELL 13 — Plot training curves
# ============================================================
def plot_training_curves(h1, h2):
    """Vẽ đường cong accuracy và loss cho cả hai stage."""
    epochs_1 = range(1, len(h1.history["accuracy"]) + 1)
    epochs_2 = range(len(h1.history["accuracy"]) + 1,
                     len(h1.history["accuracy"]) + len(h2.history["accuracy"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(epochs_1, h1.history["accuracy"], "-o", ms=3, label="train (S1)")
    axes[0].plot(epochs_1, h1.history["val_accuracy"], "-o", ms=3, label="val (S1)")
    axes[0].plot(epochs_2, h2.history["accuracy"], "-o", ms=3, label="train (S2)")
    axes[0].plot(epochs_2, h2.history["val_accuracy"], "-o", ms=3, label="val (S2)")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1.02)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_1, h1.history["loss"], "-o", ms=3, label="train (S1)")
    axes[1].plot(epochs_1, h1.history["val_loss"], "-o", ms=3, label="val (S1)")
    axes[1].plot(epochs_2, h2.history["loss"], "-o", ms=3, label="train (S2)")
    axes[1].plot(epochs_2, h2.history["val_loss"], "-o", ms=3, label="val (S2)")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    fig.suptitle(f"PPR501 — CNN Training Curves (alpha={best_alpha})", fontsize=13)
    fig.tight_layout()
    return fig


fig = plot_training_curves(history_1, history_2)
plt.show()

### Ma trận nhầm lẫn (Confusion Matrix)

Trục ngang: **Actual** (nhãn thực tế) · Trục dọc: **Predicted** (nhãn dự đoán).
Các ô trên đường chéo biểu thị những mẫu được dự đoán đúng.

In [ ]:
# ============================================================
# CELL 14 — Confusion matrix
# ============================================================
cm = confusion_matrix(y_true, y_pred_adj)

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(CLASS_NAMES, fontsize=9)
ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.set_title(f"Confusion Matrix — Test set (n={len(y_true)})")

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        color = "white" if cm[i, j] > cm.max() * 0.55 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=10)

fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()

## 9. Lưu trữ mô hình

**Mô hình, danh sách nhãn, cấu hình và kết quả đánh giá** được lưu chung vào thư mục
`outputs/car_cnn/`, để phần web app ở cell tiếp theo có thể tải lại và sử dụng.

In [ ]:
# ============================================================
# CELL 15 — Save model + artefacts
# ============================================================
OUTPUT_DIR = Path("outputs") / "car_cnn"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save(OUTPUT_DIR / "model.keras")

(OUTPUT_DIR / "labels.json").write_text(
    json.dumps({
        "class_indices": {n: i for i, n in enumerate(CLASS_NAMES)},
        "class_names": CLASS_NAMES,
    }, indent=2)
)

(OUTPUT_DIR / "meta.json").write_text(
    json.dumps({
        "architecture": model.name,
        "image_size": IMAGE_SIZE,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "priors": {n: float(PRIORS[i]) for i, n in enumerate(CLASS_NAMES)},
        "prior_alpha": float(best_alpha),
    }, indent=2)
)

(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps({
        "val_accuracy": max(history_2.history["val_accuracy"]),
        "test_accuracy": acc_adj,
        "balanced_accuracy": balanced_adj,
        "test_accuracy_no_correction": acc_raw,
        "balanced_accuracy_no_correction": balanced_raw,
        "test_loss": float(model.evaluate(test_ds, return_dict=True)["loss"]),
        "confusion_matrix": cm.tolist(),
    }, indent=2)
)

print(f"Saved to {OUTPUT_DIR.resolve()}")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")

## 10. Ứng dụng web (Gradio)

Một giao diện web cho phép tải lên ảnh xe và hiển thị xác suất dự đoán cho cả bảy lớp.

> ℹ️ Cell này mở trình duyệt và **chặn kernel cho đến khi cửa sổ được đóng**. Để tiếp
> tục chạy các cell còn lại, nhấn **Stop** (hình vuông ■) trên thanh công cụ.

In [ ]:
# ============================================================
# CELL 16 — Prediction function + Gradio app
# ============================================================
import gradio as gr
import pandas as pd

META = json.loads((OUTPUT_DIR / "meta.json").read_text())
LABELS = META["class_names"]
ALPHA = float(META["prior_alpha"])
PRIOR_VEC = np.array([META["priors"][n] for n in LABELS])

def predict_image(img):
    """
    Nhận ảnh PIL, trả về dict {class: probability} đã áp dụng prior correction.
    Scaling phải khớp lúc train: về 224x224 rồi chia 255.
    """
    if img is None:
        return {}
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)

    img = img.convert("RGB")
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE))
    arr = np.array(img).astype("float32") / 255.0
    arr = np.expand_dims(arr, axis=0)

    probs = model.predict(arr, verbose=0)[0]
    probs = apply_prior_correction(probs, PRIOR_VEC, ALPHA)

    return {LABELS[i]: float(probs[i]) for i in range(len(LABELS))}


def _ui(img):
    if img is None:
        return {}, pd.DataFrame({"Class": [], "Probability": []})
    probs = predict_image(img)
    frame = pd.DataFrame(
        {"Class": list(probs.keys()), "Probability": list(probs.values())}
    )
    return probs, frame


with gr.Blocks(title="PPR501 — Car Image Classifier") as demo:
    gr.Markdown(
        "# Car Image Classifier\n"
        "Phân loại ảnh xe thuộc 7 nhóm: Audi, Hyundai Creta, Mahindra Scorpio, "
        "Rolls Royce, Maruti Suzuki Swift, Tata Safari, Toyota Innova."
    )
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Ảnh xe")
            submit = gr.Button("Phân loại", variant="primary")
        with gr.Column():
            output = gr.Label(num_top_classes=len(LABELS), label="Kết quả")
            if hasattr(gr, "BarPlot"):
                prob_chart = gr.BarPlot(
                    x="Class", y="Probability", y_title="Probability",
                    sort="-y", label="Xác suất từng lớp",
                )
        submit.click(_ui, inputs=image_input, outputs=[output, prob_chart])

demo.queue().launch(share=False)

## 11. Kết luận

### Kết quả

Trên tập test gồm **813 ảnh** (chưa từng được dùng để huấn luyện hay validation):

| Chỉ số | Giá trị |
|--------|---------|
| Test accuracy | **~0.97** |
| Balanced accuracy | **~0.96** |
| Test loss | ~0.11 |
| Prior alpha | xác định trên tập validation |

### Các kết quả chính

1. **Khắc phục hiện tượng rò rỉ dữ liệu** — tập validation được tách biệt hoàn toàn
   khỏi tập test.
2. **Xử lý mất cân bằng dữ liệu** — class weight theo công thức effective-number kết
   hợp với prior correction.
3. **Áp dụng transfer learning** — sử dụng MobileNetV2 (ImageNet) thay cho CNN huấn
   luyện từ đầu, do tập dữ liệu chỉ có 3 352 ảnh.
4. **Huấn luyện theo hai giai đoạn** — đóng băng backbone trước, sau đó fine-tune với
   tốc độ học nhỏ hơn.
5. **Giữ BatchNorm ở trạng thái đóng băng** trong giai đoạn fine-tune để bảo toàn
   thống kê chạy của ImageNet.
6. **Ứng dụng web** — giao diện Gradio hiển thị xác suất dự đoán của cả bảy lớp.

### Hạn chế và hướng phát triển

- Quy mô tập dữ liệu còn nhỏ và chủ yếu gồm ảnh chụp từ bên ngoài xe, do đó mô hình chưa
  được kiểm thử với ảnh chụp từ trong cabin hoặc các góc máy khác thường.
- Ma trận nhầm lẫn cho thấy phần lớn lỗi còn lại tập trung ở cặp Hyundai Creta ↔
  Toyota Innova: hai dòng xe này có ngoại hình khá giống nhau ở góc chụp phía trước.
- Hướng phát triển hợp lý bao gồm bổ sung dữ liệu, áp dụng augmentation mạnh hơn và
  thử nghiệm thêm các backbone khác (ResNet50, EfficientNet) để đối chiếu.